<!--
---
PURPOSE: Run zero-label pose estimation — Facemap (eye pupil + face SVD) and
         SuperAnimal-quadruped (side body keypoints). Outputs saved to
         outputs/pose/ for use in NB07 and NB08.
REQUIRES:
  - data/raw/visual-behavior-neuropixels/1055240613/behavior_videos/eye.mp4
  - data/raw/visual-behavior-neuropixels/1055240613/behavior_videos/face.mp4
  - data/raw/visual-behavior-neuropixels/1055240613/behavior_videos/side.mp4
PRODUCES:
  - outputs/pose/session_{id}_facemap_eye.parquet   (pupil area/pos/blink)
  - outputs/pose/session_{id}_facemap_face.parquet  (SVD motion energy PCs)
  - outputs/pose/session_{id}_superanimal.h5        (body keypoints)
WHAT_NEXT: notebooks/07_Pose_to_Motifs_Feature_Engineering.ipynb
---
-->

# 06 Pose Estimation — Facemap + SuperAnimal

**Cameras and tools:**
| Camera | Tool | Labels needed | Output |
|--------|------|---------------|--------|
| `eye.mp4` | Facemap pupil tracker | 0 | pupil area, x/y position, blink |
| `face.mp4` | Facemap SVD motion energy | 0 | top-N motion energy PCs |
| `side.mp4` | SuperAnimal-quadruped (DLC) | 0 | body keypoints (paw, spine, tail) |

**Install once (in vbn-analysis env):**
```bash
pip install facemap
pip install "deeplabcut[tf]"
```


In [ ]:
# ── Kaggle / Colab setup (skip if running locally) ───────────────────────────
import os, sys
from pathlib import Path

_on_kaggle = Path("/kaggle/working").exists()
_on_colab  = Path("/content").exists()
_is_cloud  = _on_kaggle or _on_colab

if _is_cloud:
    print("Cloud environment detected — running setup...")

    BASE = Path("/kaggle/working") if _on_kaggle else Path("/content")
    REPO = BASE / "vbn-analysis"

    # 1. Clone repo
    if not REPO.exists():
        os.system(f"git clone -q https://github.com/muhsinh/vbn-analysis.git {REPO}")
        print("Repo cloned.")
    else:
        print("Repo already present — skipping clone.")

    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))
    if str(REPO / "src") not in sys.path:
        sys.path.insert(0, str(REPO / "src"))

    # 2. Install DLC 3.x + compatible deps for Kaggle (NumPy 2.4, Python 3.12)
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
    import subprocess

    _needs_install = False
    try:
        import deeplabcut as _dlc_check
        _ver = tuple(int(x) for x in _dlc_check.__version__.split(".")[:2])
        if _ver < (3, 0):
            _needs_install = True
            print(f"DLC {_dlc_check.__version__} (TF) — upgrading to 3.x ...")
        else:
            print(f"DLC {_dlc_check.__version__} already installed — skipping.")
    except Exception:
        _needs_install = True
        print("Installing DLC 3.x ...")

    if _needs_install:
        # numba>=0.64 required for NumPy 2.4 (Kaggle default). Must install first.
        subprocess.call([
            sys.executable, "-m", "pip", "install", "-q",
            "--prefer-binary", "numba>=0.64",
        ])
        # DLC with --no-deps keeps Kaggle's NumPy 2.4 intact
        subprocess.call([
            sys.executable, "-m", "pip", "install", "-q",
            "--no-deps", "--prefer-binary", "--no-build-isolation",
            "deeplabcut>=3.0.0rc1",
        ])
        # Lightweight missing deps (torch/numpy/pandas/cv2 already on Kaggle)
        subprocess.call([
            sys.executable, "-m", "pip", "install", "-q",
            "--prefer-binary", "--no-build-isolation",
            "filterpy", "ruamel.yaml", "munkres", "dlclibrary",
        ])
        print("DLC 3.x installed.")

    # 3. Download side.mp4
    SESSION_ID = 1055240613
    SIDE_MP4 = REPO / "data" / "raw" / "visual-behavior-neuropixels" / str(SESSION_ID) / "behavior_videos" / "side.mp4"

    if SIDE_MP4.exists() and SIDE_MP4.stat().st_size > 1e9:
        print(f"side.mp4 already present ({SIDE_MP4.stat().st_size/1024**3:.2f} GB) — skipping download.")
    else:
        print("Downloading side.mp4 from S3 (~2.2 GB) ...")
        SIDE_MP4.parent.mkdir(parents=True, exist_ok=True)
        import boto3
        from botocore import UNSIGNED
        from botocore.config import Config
        s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
        s3.download_file(
            "allen-brain-observatory",
            f"visual-behavior-neuropixels/raw-data/{SESSION_ID}/behavior_videos/side.mp4",
            str(SIDE_MP4),
        )
        print(f"Downloaded: {SIDE_MP4.stat().st_size/1024**3:.2f} GB")

    # T4 has 15 GB VRAM; FasterRCNN + HRNet together need ~13 GB at batch=32 → OOM.
    # batch=8 keeps peak below 10 GB with headroom for fragmentation.
    os.environ["CLOUD_DETECTOR_BATCH"]   = "8"
    os.environ["CLOUD_POSE_BATCH"]       = "8"
    os.environ["PYTORCH_ALLOC_CONF"]     = "expandable_segments:True"
    os.environ["ACCESS_MODE"]            = "manual"
    os.environ["VBN_REPO_ROOT"]          = str(REPO)
    print("Setup complete — run the SuperAnimal cell at the bottom.")
else:
    print("Local environment — skipping cloud setup.")

## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [2]:
import sys
import importlib
import inspect
from pathlib import Path

# Resolve repo root whether notebook was launched from repo root or notebooks/.
ROOT = Path.cwd().resolve()
if (ROOT / "src").exists() and (ROOT / "notebooks").exists():
    pass
elif (ROOT.parent / "src").exists() and (ROOT.parent / "notebooks").exists():
    ROOT = ROOT.parent
else:
    raise RuntimeError(f"Could not resolve project root from cwd={Path.cwd()}")

SRC = ROOT / "src"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Ensure notebook uses the local io_sessions module from this repo.
io_sessions = importlib.import_module("io_sessions")
io_sessions = importlib.reload(io_sessions)
print("io_sessions:", io_sessions.__file__)
print("get_session_bundle:", inspect.signature(io_sessions.get_session_bundle))

import os
os.environ.setdefault("ACCESS_MODE", "manual")


io_sessions: /Users/muh/projects/vbn-analysis/src/io_sessions.py
get_session_bundle: (session_id: 'int', sessions_df: 'pd.DataFrame | None' = None, *, resolve_nwb: 'bool' = True, inspect_modalities: 'bool' = True) -> 'SessionBundle'


'manual'

## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps.

In [3]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = ROOT / "notebooks" / "06_Pose_Estimation_Setup_SLEAP_or_DLC.ipynb"
header  = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))

if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

All prerequisites satisfied.


## Step 1: Configuration
Set session IDs and number of SVD components to keep from face.mp4.


In [4]:
from config import get_config
from io_sessions import load_sessions_csv, get_session_bundle
import pandas as pd
import numpy as np
from pathlib import Path

cfg = get_config()
sessions = load_sessions_csv()
SESSION_IDS = [1055240613]

N_SVD_COMPONENTS = 20   # face.mp4 motion energy PCs to keep
CAMERAS = ["eye", "face", "side"]

VIDEO_ROOT = cfg.video_cache_dir
POSE_OUT   = cfg.outputs_dir / "pose"
POSE_OUT.mkdir(parents=True, exist_ok=True)

def video_path(session_id, camera):
    return VIDEO_ROOT / str(session_id) / "behavior_videos" / f"{camera}.mp4"

# Check what videos are available
for sid in SESSION_IDS:
    for cam in CAMERAS:
        p = video_path(sid, cam)
        status = f"{p.stat().st_size/1024**3:.2f} GB" if p.exists() else "MISSING"
        print(f"  [{sid}] {cam}.mp4: {status}")


  [1055240613] eye.mp4: 2.23 GB
  [1055240613] face.mp4: 2.23 GB
  [1055240613] side.mp4: 2.23 GB


In [7]:
# ── Facemap: eye pupil tracking + face SVD ─────────────────────────────────
try:
    import facemap
    from facemap import process
    _facemap_available = True
except ImportError:
    _facemap_available = False
    print("Facemap not installed. Run: pip install facemap")

from timebase import write_parquet_with_timebase
from config import make_provenance

if _facemap_available:
    for sid in SESSION_IDS:
        # ── eye.mp4: pupil tracking ──────────────────────────────────────────
        eye_vid = video_path(sid, "eye")
        eye_out = POSE_OUT / f"session_{sid}_facemap_eye.parquet"

        if eye_out.exists():
            print(f"[{sid}] eye facemap already cached — skipping")
        elif not eye_vid.exists():
            print(f"[{sid}] eye.mp4 not found — download via NB01 cell 11")
        else:
            print(f"[{sid}] Running Facemap pupil tracking on eye.mp4 ...")
            try:
                process.run(
                    filenames=[[str(eye_vid)]],
                    sbin=4,
                    motSVD=False,
                    movSVD=False,
                    savepath=str(POSE_OUT),
                )
                proc = np.load(POSE_OUT / "eye_proc.npy", allow_pickle=True).item()
                pups = proc.get("pupil", [{}])
                pupil_data = pups[0] if pups else {}
                n_frames = len(pupil_data.get("area", []))
                t_approx = np.arange(n_frames) / 30.0
                com = pupil_data.get("com_smooth", pupil_data.get("com", np.full((n_frames, 2), np.nan)))
                df_eye = pd.DataFrame({
                    "frame": np.arange(n_frames),
                    "t": t_approx,
                    "pupil_area": pupil_data.get("area_smooth", pupil_data.get("area", np.full(n_frames, np.nan))),
                    "pupil_x": com[:, 0] if hasattr(com, "shape") and com.ndim == 2 else np.nan,
                    "pupil_y": com[:, 1] if hasattr(com, "shape") and com.ndim == 2 else np.nan,
                })
                blinks = proc.get("blink", [])
                if blinks and blinks[0] is not None:
                    df_eye["blink"] = np.asarray(blinks[0]).astype(bool)
                write_parquet_with_timebase(
                    df_eye, eye_out,
                    provenance=make_provenance(sid, "facemap_eye"),
                    required_columns=["frame"],
                )
                print(f"  Saved: {eye_out} ({len(df_eye)} frames)")
            except Exception as e:
                print(f"  ERROR running Facemap on eye.mp4: {e}")

        # ── face.mp4: SVD motion energy ──────────────────────────────────────
        face_vid = video_path(sid, "face")
        face_out = POSE_OUT / f"session_{sid}_facemap_face.parquet"

        if face_out.exists():
            print(f"[{sid}] face SVD already cached — skipping")
        elif not face_vid.exists():
            print(f"[{sid}] face.mp4 not found — download via NB01 cell 11")
        else:
            print(f"[{sid}] Running Facemap SVD on face.mp4 ...")
            try:
                process.run(
                    filenames=[[str(face_vid)]],
                    sbin=4,
                    motSVD=True,
                    movSVD=False,
                    savepath=str(POSE_OUT),
                )
                proc = np.load(POSE_OUT / "face_proc.npy", allow_pickle=True).item()
                svd_list = proc.get("motSVD", [None])
                mot_svd = svd_list[0] if svd_list else None
                if mot_svd is not None:
                    n_frames = mot_svd.shape[0]
                    n_keep = min(N_SVD_COMPONENTS, mot_svd.shape[1])
                    t_approx = np.arange(n_frames) / 30.0
                    svd_cols = {f"face_svd_{i}": mot_svd[:, i] for i in range(n_keep)}
                    df_face = pd.DataFrame({"frame": np.arange(n_frames), "t": t_approx, **svd_cols})
                    write_parquet_with_timebase(
                        df_face, face_out,
                        provenance=make_provenance(sid, "facemap_face_svd"),
                        required_columns=["frame"],
                    )
                    print(f"  Saved: {face_out} ({n_frames} frames, {n_keep} SVD components)")
                else:
                    print("  WARNING: motSVD not in facemap output")
            except Exception as e:
                print(f"  ERROR running Facemap on face.mp4: {e}")


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


[1055240613] Running Facemap pupil tracking on eye.mp4 ...
Computing subsampled mean...
Computed subsampled mean at 2.31s
Computing subsampled SVD...
computed svd chunk 0 / 15, time  0.38sec
computed svd chunk 1 / 15, time  0.83sec
computed svd chunk 2 / 15, time  1.31sec
computed svd chunk 3 / 15, time  1.72sec
computed svd chunk 4 / 15, time  2.14sec
computed svd chunk 5 / 15, time  2.55sec
computed svd chunk 6 / 15, time  2.96sec
computed svd chunk 7 / 15, time  3.36sec
computed svd chunk 8 / 15, time  3.74sec
computed svd chunk 9 / 15, time  4.13sec
computed svd chunk 10 / 15, time  4.51sec
computed svd chunk 11 / 15, time  4.90sec
computed svd chunk 12 / 15, time  5.29sec
computed svd chunk 13 / 15, time  5.70sec
computed svd chunk 14 / 15, time  6.08sec
Computed subsampled SVD at 8.40s
Computing ROIs and/or motSVD/movSVD
computed video chunk 0 / 1152, time  9.34sec
computed video chunk 10 / 1152, time  18.20sec
computed video chunk 20 / 1152, time  27.64sec
computed video chunk 3

In [5]:
import numpy as np, pandas as pd                                                                                                                                                                                   
from pathlib import Path                                     
from timebase import write_parquet_with_timebase
from config import make_provenance
                                                                                                                                                                                                                     
POSE_OUT = ROOT / "outputs/pose"
sid = 1055240613                                                                                                                                                                                                   
                                                                                                                                                                                                                     
POSE_OUT = ROOT / "outputs/pose"                                                                                                                                                                                   
sid = 1055240613                                                                                                                                                                                                   
                                                                                                                                                                                                                     
proc = np.load(ROOT / "outputs/pose/face_proc.npy", allow_pickle=True).item()                                                                                                                                       
mot_svd = proc["motSVD"][0]
n = mot_svd.shape[0]                                                                                                                                                                                               
df_face = pd.DataFrame({"frame": np.arange(n), "t": np.arange(n)/30.0,
    **{f"face_svd_{i}": mot_svd[:,i] for i in range(min(20, mot_svd.shape[1]))}})                                                                                                                                  
write_parquet_with_timebase(df_face, POSE_OUT / f"session_{sid}_facemap_face.parquet",
    provenance=make_provenance(sid, "facemap_face_svd"), required_columns=["frame"])                                                                                                                               
print(f"face: {n} frames, {min(20, mot_svd.shape[1])} SVD components saved")

face: 575523 frames, 20 SVD components saved


In [ ]:
# ── SuperAnimal-quadruped: body keypoints from side.mp4 ─────────────────────
import os, sys, subprocess, shutil, time, textwrap
import torch
import pandas as pd
import numpy as np
from pathlib import Path

# ── Resolve ROOT ─────────────────────────────────────────────────────────────
if "ROOT" not in dir():
    _repo_env = os.environ.get("VBN_REPO_ROOT")
    if _repo_env:
        ROOT = Path(_repo_env)
    else:
        ROOT = Path.cwd().resolve()
        if not (ROOT / "src").exists():
            ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT))
    sys.path.insert(0, str(ROOT / "src"))

os.environ["HF_HOME"]     = str(ROOT / "data" / "hf_cache")
os.environ["HF_XET_CACHE"] = str(ROOT / "data" / "hf_cache" / "xet")
(ROOT / "data" / "hf_cache" / "xet").mkdir(parents=True, exist_ok=True)

SESSION_IDS = [1055240613]
POSE_OUT = ROOT / "outputs" / "pose"
POSE_OUT.mkdir(parents=True, exist_ok=True)

def video_path(session_id, camera):
    return ROOT / "data" / "raw" / "visual-behavior-neuropixels" / str(session_id) / "behavior_videos" / f"{camera}.mp4"

# ── Import DLC ───────────────────────────────────────────────────────────────
try:
    import deeplabcut as dlc
    print(f"DLC {dlc.__version__} loaded OK")
    _dlc_available = True
except Exception as e:
    _dlc_available = False
    print(f"DLC import failed: {type(e).__name__}: {e}")

if not _dlc_available:
    raise SystemExit("Fix DLC import before continuing.")

# ── GPU inventory ─────────────────────────────────────────────────────────────
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if n_gpus > 0:
    for i in range(n_gpus):
        gb = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"  cuda:{i}  {torch.cuda.get_device_name(i)}  ({gb:.1f} GB)")
    _device = "cuda"
elif torch.backends.mps.is_available():
    # Patch DLC's resolve_device — it blocks HRNet from using MPS
    import deeplabcut.pose_estimation_pytorch.utils as _dlc_pt_utils
    def _patched_resolve_device(model_config):
        dev = model_config.get("device", "auto")
        if dev == "auto":
            if torch.cuda.is_available():   return "cuda"
            if torch.backends.mps.is_available(): return "mps"
            return "cpu"
        return dev
    _dlc_pt_utils.resolve_device = _patched_resolve_device
    _device = "mps"
    n_gpus = 1
    print("Using device: mps (patched resolve_device)")
else:
    _device = "cpu"
    n_gpus = 1
    print("No GPU — using cpu (will be slow)")

# Per-GPU batch sizes: each T4 gets its own 15 GB, so 16 is safe when independent
DETECTOR_BATCH = int(os.environ.get("CLOUD_DETECTOR_BATCH", 8))
POSE_BATCH     = int(os.environ.get("CLOUD_POSE_BATCH", 8))
print(f"n_gpus={n_gpus}, detector_batch={DETECTOR_BATCH}, pose_batch={POSE_BATCH}")

# ── Runner script written to /tmp for subprocess calls ───────────────────────
RUNNER = "/tmp/dlc_runner.py"
Path(RUNNER).write_text(textwrap.dedent("""
    import sys, os
    os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
    device, video, dest, pose_b, det_b, repo = sys.argv[1:7]
    sys.path.insert(0, repo)
    sys.path.insert(0, os.path.join(repo, "src"))
    import deeplabcut as dlc
    gpu_id = os.environ.get("CUDA_VISIBLE_DEVICES", "?")
    print(f"[cuda:{gpu_id}] Starting {os.path.basename(video)} ...", flush=True)
    dlc.video_inference_superanimal(
        videos=[video],
        superanimal_name="superanimal_quadruped",
        model_name="hrnet_w32",
        detector_name="fasterrcnn_resnet50_fpn_v2",
        scale_list=[200],
        videotype="mp4",
        dest_folder=dest,
        create_labeled_video=False,
        plot_trajectories=False,
        batch_size=int(pose_b),
        detector_batch_size=int(det_b),
        device=device,
    )
    print(f"[cuda:{gpu_id}] Done.", flush=True)
"""))

# ── Main inference loop ───────────────────────────────────────────────────────
for sid in SESSION_IDS:
    side_vid = video_path(sid, "side")
    sa_out   = POSE_OUT / f"session_{sid}_superanimal.h5"

    if sa_out.exists():
        print(f"[{sid}] SuperAnimal already cached — skipping")
        continue
    if not side_vid.exists():
        print(f"[{sid}] side.mp4 not found at {side_vid}")
        continue

    # ── Multi-GPU path: split video → run in parallel → merge H5 ─────────────
    if n_gpus >= 2:
        print(f"[{sid}] Multi-GPU mode: splitting video across {n_gpus} GPUs ...")

        import cv2
        cap = cv2.VideoCapture(str(side_vid))
        n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps     = cap.get(cv2.CAP_PROP_FPS) or 30.0
        cap.release()
        mid_sec = (n_total // 2) / fps
        print(f"  Total frames: {n_total}, fps: {fps:.2f}, split at {mid_sec:.1f}s")

        parts = ["/tmp/side_part0.mp4", "/tmp/side_part1.mp4"]
        dests = ["/tmp/dlc_out0", "/tmp/dlc_out1"]
        for d in dests:
            Path(d).mkdir(exist_ok=True)

        # Fast split with stream copy (no re-encode, ~seconds)
        print("  Splitting with ffmpeg (-c copy) ...")
        r0 = subprocess.call([
            "ffmpeg", "-y", "-i", str(side_vid),
            "-t", str(mid_sec), "-c", "copy", parts[0],
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        r1 = subprocess.call([
            "ffmpeg", "-y", "-ss", str(mid_sec), "-i", str(side_vid),
            "-c", "copy", parts[1],
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if r0 != 0 or r1 != 0:
            raise RuntimeError(f"ffmpeg split failed (exit {r0}, {r1})")
        for i, p in enumerate(parts):
            sz = Path(p).stat().st_size / 1024**2
            print(f"  part{i}: {sz:.0f} MB")

        # Launch both GPUs in parallel
        procs = []
        envs  = []
        for i in range(2):
            e = os.environ.copy()
            e["CUDA_VISIBLE_DEVICES"] = str(i)
            p = subprocess.Popen(
                [sys.executable, RUNNER,
                 "cuda", parts[i], dests[i],
                 str(POSE_BATCH), str(DETECTOR_BATCH), str(ROOT)],
                env=e,
            )
            procs.append(p)
            print(f"  GPU {i} started (PID {p.pid})")

        # Wait with progress ticks
        t0 = time.time()
        while any(p.poll() is None for p in procs):
            time.sleep(30)
            elapsed = (time.time() - t0) / 60
            alive = [i for i, p in enumerate(procs) if p.poll() is None]
            print(f"  ... {elapsed:.0f} min elapsed, GPUs still running: {alive}")
        for i, p in enumerate(procs):
            print(f"  GPU {i} exit code: {p.returncode}")

        # Merge H5 outputs
        print("  Merging H5 outputs ...")
        dfs = []
        for i, dest in enumerate(dests):
            h5s = list(Path(dest).glob("*.h5"))
            if not h5s:
                raise RuntimeError(f"No H5 found in {dest} for GPU {i}")
            df = pd.read_hdf(h5s[0])
            if i > 0:
                # Count frames in part 0 to compute offset
                cap0 = cv2.VideoCapture(parts[0])
                n_part0 = int(cap0.get(cv2.CAP_PROP_FRAME_COUNT))
                cap0.release()
                df.index = df.index + n_part0
            dfs.append(df)
            print(f"  part{i}: {len(df)} frames")

        merged = pd.concat(dfs).sort_index()
        merged.to_hdf(str(sa_out), key="df_with_missing", mode="w")
        print(f"  Merged {len(merged)} frames → {sa_out}")

        # Cleanup temp files
        for p in parts:
            Path(p).unlink(missing_ok=True)

    # ── Single-GPU path ───────────────────────────────────────────────────────
    else:
        print(f"[{sid}] Running SuperAnimal-quadruped (single GPU) ...")
        try:
            dlc.video_inference_superanimal(
                videos=[str(side_vid)],
                superanimal_name="superanimal_quadruped",
                model_name="hrnet_w32",
                detector_name="fasterrcnn_resnet50_fpn_v2",
                scale_list=[200],
                videotype="mp4",
                dest_folder=str(POSE_OUT),
                create_labeled_video=False,
                plot_trajectories=False,
                batch_size=POSE_BATCH,
                detector_batch_size=DETECTOR_BATCH,
                device=_device,
            )
            h5_candidates = list(POSE_OUT.glob(f"{side_vid.stem}*superanimal*.h5"))
            if not h5_candidates:
                h5_candidates = list(side_vid.parent.glob(f"{side_vid.stem}*superanimal*.h5"))
            if h5_candidates:
                if str(h5_candidates[0]) != str(sa_out):
                    shutil.move(str(h5_candidates[0]), str(sa_out))
                print(f"  Saved: {sa_out}")
            else:
                print(f"  WARNING: Could not find output .h5 — check {POSE_OUT}")
        except Exception as e:
            print(f"  ERROR: {type(e).__name__}: {e}")

print("\nDone. Download outputs/pose/session_*_superanimal.h5 to use in NB07.")